# Pipeline d'Imagerie du Ciel Profond en Astrophotographie
*Par: Nicolas Payot, Gabriel Missael Barco, Auriane Thilloy, Olivia Pereira*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Traditional_Pipeline_DSO.ipynb)      [![View on GitHub](https://img.shields.io/badge/View_on-GitHub-black?logo=github)](https://github.com/GabrielMissael/super-resolution-workshop)

Ce notebook explore le pipeline de traitement pour l'imagerie du ciel profond (DSO - Deep-Sky Object), comme les galaxies et les nébuleuses. Contrairement à l'imagerie planétaire qui utilise des poses très courtes, le ciel profond nécessite des poses longues pour capturer des objets très peu lumineux.

Nous verrons les étapes essentielles :
1.  **Calibration** : Correction des défauts du capteur avec des images spéciales (darks, flats).
2.  **Alignement** : Recalage des images en se basant sur la position des étoiles.
3.  **Dématriçage (Debayering)** : Reconstruction des couleurs à partir des données brutes du capteur.
4.  **Empilement (Stacking)** : Combinaison des images pour améliorer le rapport signal/bruit.
5.  **Amélioration** : Réduction du bruit et amélioration des détails.

In [38]:
import sys
sys.path.append('super-resolution-workshop')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ImageNormalize, AsinhStretch
from ipywidgets import interact, FloatSlider

from src.process.dso_utils import normalize_flat, calibrate_frame, make_master_frame, percentile_normalize, crop
from src.process.star_align import visualize_matches, compute_transforms_relative_to_reference, visualize_image_positions, warp_images_to_reference
from src.demosaic.debayer import debayer_image

## 1. Chargement des Données Brutes (Images "Lights")

La première étape consiste à charger nos images brutes, aussi appelées **"lights"**. Ce sont les poses longues qui contiennent l'objet céleste que nous voulons photographier (ici, la galaxie NGC 7331).

Chaque image est affectée par des signaux non désirés (bruit thermique, poussières, etc.). L'objectif du traitement est de nettoyer ces images pour ne garder que le signal de la galaxie.

La cellule ci-dessous charge les premières images de notre séquence pour visualiser leur contenu.

In [ ]:
def frame_file(n):
    # Prefer files named with frame_NNNNN pattern; otherwise use first 4 files
    candidate = data_dir / f'frame_{n:05d}.fits'
    if candidate.exists():
        return candidate
    # fallback: use index (n-1) from sorted list if available
    idx = n - 1
    if idx < len(files):
        return files[idx]
    return None


# Ensure repo root is on path
proj_root = Path('..').resolve()
if proj_root.match("/"):
    proj_root = Path('super-resolution-workshop').resolve()
sys.path.insert(0, str(proj_root))
data_dir = proj_root / 'data' / 'ngc7331_crops'
# print('Looking for FITS in', data_dir)
if not data_dir.exists():
    raise FileNotFoundError(f'Data directory not found: {data_dir}')
else:
    files = sorted(list(data_dir.glob('*.fits')))

chosen = []
for n in [1, 2, 3, 4]:
    p = frame_file(n)
    if p is None:
        print(f'Frame {n} not found')
    else:
        chosen.append(p)
if not chosen:
    raise FileNotFoundError('No FITS files found in data directory')

# Load data once to make the slider responsive
raw_frames_data = []
for p in chosen:
    with fits.open(str(p)) as hdul:
        raw_frames_data.append(hdul[0].data.astype(float))

# Create the plot structure once
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.ravel()
images = []
for i, (ax, p) in enumerate(zip(axes, chosen)):
    # Display initial image
    img = ax.imshow(raw_frames_data[i], cmap='gray', norm=ImageNormalize(stretch=AsinhStretch(a=0.005)))
    images.append(img)
    ax.set_title(p.name)
    ax.axis('off')

for ax in axes[len(chosen):]:
    ax.axis('off')
plt.suptitle('NGC7331 crops: frames 1-4')
plt.tight_layout(rect=[0, 0, 1, 0.96])

@interact(a=FloatSlider(min=0.001, max=0.2, step=0.001, value=0.005, description='Asinh Stretch (a)', continuous_update=False))
def update_stretch(a):
    for img in images:
        img.set_norm(ImageNormalize(stretch=AsinhStretch(a=a)))
    fig.canvas.draw_idle()

## 2. Calibration avec les Darks

Les **"darks"** sont des images prises avec le même temps de pose et la même température que les images "lights", mais avec le capuchon sur le télescope.

**Leur but ?** Capturer le **bruit thermique** du capteur. Ce bruit, qui dépend de la température et du temps de pose, ajoute des pixels lumineux parasites. En soustrayant l'image "dark" de nos images brutes, on élimine une grande partie de ce bruit indésirable.

Ici, nous chargeons une image "master dark", qui est une moyenne de plusieurs darks individuels pour réduire le bruit aléatoire qu'ils contiennent.

In [ ]:
dark_candidates = sorted((data_dir.glob('*master*dark*.fits')) or [])
# also try any dark*.fits if master not found
if not dark_candidates:
    dark_candidates = sorted(list(data_dir.glob('*dark*.fits')))
if not dark_candidates:
    raise FileNotFoundError('No dark frames found in data directory')

# use first dark candidate
p = dark_candidates[0]
with fits.open(str(p)) as hdul:
    arr = hdul[0].data.astype(float)
vmin, vmax = np.percentile(arr, (1, 99))
plt.figure(figsize=(8, 6))
plt.imshow(arr, cmap='gray', vmin=vmin, vmax=vmax)
plt.title('Master dark: ' + p.name)
plt.axis('off')
plt.show()

## 3. Calibration avec les Flats

Les **"flats"** sont des images d'un fond uniformément éclairé (comme un ciel au crépuscule ou un écran spécial).

**Leur but ?** Corriger deux problèmes majeurs :
1.  **Le vignettage** : l'assombrissement des coins de l'image.
2.  **Les poussières** : les petites ombres créées par des poussières sur le capteur ou les optiques.

En divisant nos images par l'image "flat", on uniformise la luminosité et on fait disparaître ces défauts.

In [ ]:
flat_candidates = sorted(list(data_dir.glob('*master*flat*.fits')))
if not flat_candidates:
    flat_candidates = sorted(list(data_dir.glob('*flat*.fits')))[0:8] if list(data_dir.glob('*flat*.fits')) else []
if not flat_candidates:
    raise FileNotFoundError('No flat frames found in data directory')

p = flat_candidates[0]
with fits.open(str(p)) as hdul:
    arr = hdul[0].data.astype(float)
vmin, vmax = np.percentile(arr, (1, 99))
plt.figure(figsize=(8, 6))
plt.imshow(arr, cmap='gray', vmin=vmin, vmax=vmax)
plt.title('Flat: ' + p.name)
plt.axis('off')
plt.show()

## 4. Processus de Calibration

Maintenant que nous avons nos images "lights", "darks" et "flats", nous pouvons calibrer chaque image brute.

La formule de calibration est la suivante :

**Image Calibrée = (Image Brute - Master Dark) / (Master Flat Normalisé)**

- **Normaliser le flat** : On divise chaque pixel du master flat par la valeur médiane de l'image pour que sa valeur moyenne soit proche de 1. Cela évite de modifier la luminosité globale de l'image brute.
- **Appliquer la calibration** : On applique la formule à chaque image "light" de notre séquence.

La cellule suivante définit les fonctions nécessaires et applique ce processus à nos images.

In [ ]:
with fits.open(str(flat_candidates[0])) as hdul:
    fits_data = hdul[0].data.astype(float)
    flat_norm = normalize_flat(fits_data)

with fits.open(str(dark_candidates[0])) as hdul:
    master_dark = hdul[0].data.astype(float) / 2 ** 16

calibrated_frames = []
for n in range(1, 6):
    p = frame_file(n)
    with fits.open(str(p)) as hdul:
        raw = hdul[0].data.astype(float) / 2 ** 16
    calibrated = calibrate_frame(raw, master_dark, flat_norm)
    calibrated_frames.append(calibrated)

# Load the raw frame for comparison
with fits.open(str(frame_file(2))) as hdul:
    raw = hdul[0].data.astype(float) / 2 ** 16

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define a common normalization for visualization
norm = ImageNormalize(stretch=AsinhStretch(a=0.01))
cmap = "gray"

# 1. Raw Frame
im1 = axes[0].imshow(percentile_normalize(raw, 20, 99.9), norm=norm, cmap=cmap)
axes[0].set_title('Raw Frame 2')
axes[0].axis('off')
# fig.colorbar(im1, ax=axes[0], shrink=0.8)

# 2. Calibrated Frame
im2 = axes[1].imshow(percentile_normalize(calibrated_frames[1], 20, 99.9), norm=norm, cmap=cmap)
axes[1].set_title('Calibrated Frame 2')
axes[1].axis('off')
# fig.colorbar(im2, ax=axes[1], shrink=0.8)

# 3. Difference (Calibrated - Raw)
difference = calibrated_frames[1] - raw
im3 = axes[2].imshow(percentile_normalize(difference, 20, 99.9), norm=norm, cmap=cmap)
axes[2].set_title('Difference (Calibrated - Raw)')
axes[2].axis('off')
# fig.colorbar(im3, ax=axes[2], shrink=0.8)

plt.suptitle('Comparison of Raw vs. Calibrated Frame')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 5. Alignement par les Étoiles

À cause de la rotation de la Terre et des petites imperfections de la monture du télescope, chaque image est légèrement décalée par rapport aux autres. Avant de les empiler, il faut les aligner parfaitement.

Pour cela, on utilise les étoiles comme points de référence :
1.  **Détection d'étoiles** : Un algorithme détecte les étoiles les plus brillantes dans chaque image.
2.  **Choix d'une image de référence** : On choisit l'image la plus nette de la série comme base pour l'alignement.
3.  **Calcul des transformations** : Pour chaque autre image, l'algorithme compare la position de ses étoiles à celles de l'image de référence. Il calcule ensuite la transformation géométrique (translation, rotation) nécessaire pour superposer parfaitement les étoiles.

La cellule ci-dessous détecte les étoiles et calcule ces transformations.

In [ ]:
detect_kwargs = dict(threshold_sigma=5.0, min_distance=6, max_stars=100)
match_kwargs = dict(max_match_dist=12.0, ransac_thresh=3.0)

res = compute_transforms_relative_to_reference(calibrated_frames, ref_idx=0,
                                               detect_kwargs=detect_kwargs,
                                               match_kwargs=match_kwargs)

In [ ]:
transforms = res['transforms']
star_coords = res['star_coords']
matches = res['matches']

# Plot reference frame with detected stars
norm = ImageNormalize(stretch=AsinhStretch(a=0.01))
plt.figure(figsize=(8, 8))
plt.imshow(percentile_normalize(calibrated_frames[0], 20, 99.9), cmap='gray', norm=norm)
if len(star_coords[0]) > 0:
    plt.scatter(star_coords[0][:, 0], star_coords[0][:, 1], s=30, edgecolor='yellow', facecolor='none')
plt.title('Reference frame (detected stars)')
plt.axis('off')
plt.show()

### 5.1 Comprendre les Lignes Vertes

Pour s'assurer que les étoiles sont correctement associées entre deux images, l'algorithme crée un "pattern" (ou une forme) en utilisant un groupe d'étoiles sur l'image de référence. Ce pattern est géométriquement complexe (par exemple, un triangle ou un polygone).

Ensuite, il recherche ce même pattern sur l'image cible. Comme le pattern est unique, il y a très peu de chances de le retrouver par hasard. Si l'algorithme réussit à répliquer ce pattern sur l'image cible, on peut être quasiment certain que les étoiles ont été correctement identifiées et que la transformation calculée est la bonne.

Les lignes vertes représentent les connexions entre les étoiles qui forment ce pattern validé.

In [ ]:
n_plots = len(calibrated_frames) - 1
if n_plots > 0:
    cols = 2
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = np.array(axes).flatten()

    for i in range(1, len(calibrated_frames)):
        ax = axes[i-1]
        matched = matches[i]
        M = transforms[i]
        other_pts = star_coords[i] if star_coords[i] is not None else np.zeros((0, 2))
        ref_pts = star_coords[0] if star_coords[0] is not None else np.zeros((0, 2))
        visualize_matches(ax, percentile_normalize(calibrated_frames[0], 20, 99),
                          percentile_normalize(calibrated_frames[i], 20, 99), ref_pts, other_pts, M=M,
                          matched_src=(matched.get('matched_src') if matched else None),
                          matched_dst=(matched.get('matched_dst') if matched else None),
                          inliers=(matched.get('inliers') if matched else None))
        ax.set_title(f'Reference vs image {i}')


    plt.suptitle('Matches between Reference Frame and Other Frames', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.legend(loc='upper right')
    plt.show()

### 5.3 Visualisation de l'Orientation des Images

Cette visualisation montre comment chaque image est positionnée par rapport à l'image de référence. Les décalages (translation et rotation) sont souvent très faibles, de l'ordre de quelques pixels.

Pour mieux voir ces décalages, la visualisation **exagère les transformations par un facteur 100**. Cela permet de voir clairement dans quelle direction chaque image a bougé.

- Les **carrés colorés** représentent la position originale des images.

In [ ]:
def exaggerate_transforms(transforms, trans_scale=1000.0, rot_scale=100.0):
    """Return a list of transforms where translations are multiplied by trans_scale and
    rotation/scale deviations from identity are multiplied by rot_scale.
    This exaggerates tiny shears/rotations as well as translations.
    """
    out = []
    for M in transforms:
        if M is None:
            out.append(None)
            continue
        a, b, tx = float(M[0, 0]), float(M[0, 1]), float(M[0, 2])
        c, d, ty = float(M[1, 0]), float(M[1, 1]), float(M[1, 2])
        # deviations from identity
        a_new = 1.0 + (a - 1.0) * float(rot_scale)
        d_new = 1.0 + (d - 1.0) * float(rot_scale)
        b_new = b * float(rot_scale)
        c_new = c * float(rot_scale)
        tx_new = tx * float(trans_scale)
        ty_new = ty * float(trans_scale)
        out.append(np.array([[a_new, b_new, tx_new], [c_new, d_new, ty_new]]))
    return out


shapes = [calibrated_frames[i].shape for i in range(len(calibrated_frames))]
labels = [f'Img {i}' if i != 0 else "Reference" for i in range(len(calibrated_frames))]

trans_ex2 = exaggerate_transforms(transforms, trans_scale=100, rot_scale=100)
visualize_image_positions(trans_ex2, shapes, labels=labels, translate_scale=1.0, show_original=True, auto_zoom=True)

### 5.3 Visualisation de l'Alignement

Pour vérifier que l'alignement a bien fonctionné, on peut superposer les images et tracer des lignes entre les étoiles correspondantes.

- Les **points jaunes** sont les étoiles de l'image de référence.
- Les **points cyan** sont les étoiles de l'image à aligner, après transformation.
- Les **lignes vertes** connectent les paires d'étoiles qui ont été correctement associées (les "inliers").

Si l'alignement est réussi, les points cyan et jaunes devraient être très proches les uns des autres.

In [ ]:
def show_match_pair(idx, ref_idx=0, composite=True):
    def transform_points(M, pts):
        if M is None or len(pts) == 0:
            return np.zeros((0, 2))
        pts_h = np.concatenate([pts, np.ones((len(pts), 1))], axis=1)  # (N,3)
        T = np.vstack([M, [0.0, 0.0, 1.0]])
        tr = (T @ pts_h.T).T[:, :2]
        return tr

    # Prepare images (normalized for display)
    imgL = percentile_normalize(calibrated_frames[ref_idx], 20, 99)
    imgR = percentile_normalize(calibrated_frames[idx], 20, 99)
    h, w = imgL.shape
    bar_width = 20  # Width of white space in pixels
    separator = np.ones((h, bar_width), dtype=imgL.dtype)  # white separator (value=1 for normalized images)
    comp = np.hstack([imgL, separator, imgR])  # add white bar between

    # Matched points
    matched = matches[idx]
    if matched is None:
        print(f"No matches for image {idx}")
        return
    src_pts = matched.get('matched_src')
    dst_pts = matched.get('matched_dst')
    inliers = matched.get('inliers')
    M = transforms[idx]

    if src_pts is None or dst_pts is None or len(src_pts) == 0:
        print(f"No matched star pairs for image {idx}")
        return

    src_tr = transform_points(M, src_pts) if M is not None else np.zeros_like(src_pts)
    # Adjust x-coordinates for right image to account for the separator bar as well
    src_shifted = src_pts.copy()
    src_shifted[:, 0] += w + bar_width

    cmap_in = 'lime'
    cmap_out = 'red'
    color_list = [cmap_in if (inliers is not None and inliers[i]) else cmap_out for i in range(len(src_pts))]

    res_vecs = dst_pts - src_tr
    res_norms = np.linalg.norm(res_vecs, axis=1)

    if composite:
        fig = plt.figure(figsize=(14, 6))
        ax_comp = fig.add_subplot(1, 1, 1)
        ax_comp.imshow(comp, cmap='gray', vmin=0, vmax=1)
        ax_comp.scatter(dst_pts[:, 0], dst_pts[:, 1], s=20, facecolors='none', edgecolors='yellow', label='ref stars')
        ax_comp.scatter(src_shifted[:, 0], src_shifted[:, 1], marker='+', color='cyan', s=20, label='other stars')
        for i in range(len(src_pts)):
            ax_comp.plot(
                [dst_pts[i, 0], src_shifted[i, 0]], [dst_pts[i, 1], src_shifted[i, 1]],
                color=color_list[i], linewidth=0.5, alpha=0.25
            )
        ax_comp.set_title(f'Composite: ref (left) vs img {idx} (right) - green=inlier, red=outlier')
        ax_comp.axis('off')
        ax_comp.legend(loc='upper right')
    else:
        fig = plt.figure(figsize=(8, 6))
        ax_over = fig.add_subplot(1, 1, 1)
        ax_over.imshow(imgL, cmap='gray', vmin=0, vmax=1)
        ax_over.scatter(dst_pts[:, 0], dst_pts[:, 1], s=40, edgecolors='yellow', facecolors='none', label='ref stars')
        if len(src_tr) > 0:
            ax_over.scatter(src_tr[:, 0], src_tr[:, 1], marker='+', color='cyan', s=40, label='src->ref (transformed)')
        for i in range(len(src_tr)):
            col = cmap_in if (inliers is not None and inliers[i]) else cmap_out
            ax_over.arrow(src_tr[i, 0], src_tr[i, 1], res_vecs[i, 0], res_vecs[i, 1],
                          color=col, head_width=1.5, length_includes_head=True, alpha=0.8)
        ax_over.set_title(
            f'Overlay: transformed source (cyan) -> ref (yellow). Mean residual: {res_norms.mean():.3f}px')
        ax_over.axis('off')

        ax_hist = fig.add_axes([0.66, 0.08, 0.22, 0.18])
        ax_hist.hist(res_norms, bins=20, color='dodgerblue', alpha=0.8)
        ax_hist.set_xlabel('res (px)')
        ax_hist.set_ylabel('count')
        ax_hist.set_title('Residual lengths')

        plt.tight_layout()
    plt.show()


show_match_pair(4, ref_idx=0)

## 6. Dématriçage (Debayering)

Les capteurs couleur en astronomie utilisent une **matrice de Bayer** (souvent RGGB), où chaque pixel ne voit qu'une seule couleur (rouge, vert ou bleu). L'image brute est donc une mosaïque de pixels monochromes.

Le **dématriçage** est le processus qui reconstruit une image en couleur (RGB) à partir de cette mosaïque. Pour chaque pixel, l'algorithme interpole les deux couleurs manquantes en se basant sur les pixels voisins.

Nous appliquons cette étape après la calibration et avant l'alignement final pour obtenir des images en couleur.

In [ ]:
def debayer_images(data):
    debayered = []
    for i in range(len(data)):
        temp = data[i]
        temp = (temp - temp.min()) / (temp.max() - temp.min()) * 255

        debayered.append(debayer_image(temp.astype(np.uint8), pattern="RGGB", method="vng"))
    return np.array(debayered)


debayered_calibrated_frames = debayer_images(calibrated_frames)

In [ ]:
stretch = ImageNormalize(stretch=AsinhStretch(a=0.01))

n_images = len(debayered_calibrated_frames)
cols = 3
rows = (n_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
axes = np.array(axes).flatten()

for i in range(n_images):
    ax = axes[i]
    image = crop(stretch(percentile_normalize(debayered_calibrated_frames[i], 60, 100)), size=800)
    ax.imshow(image - image.min())
    ax.set_title(f'Debayered image {i}')
    ax.axis('off')

for i in range(n_images, len(axes)):
    axes[i].axis('off')

plt.suptitle('Debayered Calibrated Frames', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 7. Application des Transformations (Warping) et Empilement

Une fois les transformations calculées, on les applique à chaque image calibrée pour les recaler sur l'image de référence. C'est l'étape de **"warping"**.

Ensuite, on procède à l'**empilement** (ou "stacking") :
- On superpose toutes les images alignées.
- On calcule la valeur médiane ou la moyenne de chaque pixel.

L'empilement a deux avantages majeurs :
1.  **Réduction du bruit** : Le bruit aléatoire est lissé, ce qui donne une image beaucoup plus propre.
2.  **Augmentation du signal** : Le signal faible de la galaxie, présent dans chaque image, est renforcé.

Cela permet de révéler des détails et des extensions de la galaxie qui étaient invisibles dans les images individuelles.

In [ ]:
# Choose which reference index to warp into (0..N-1). Use the same index used when computing transforms if applicable.
REF_IDX = 0
# You can change interpolation to 'nearest' or 'cubic'
warped_debayered_calibrated_frames = warp_images_to_reference(debayered_calibrated_frames, transforms, ref_idx=REF_IDX,
                                                              interpolation='cubic',
                                                              border_value=0.0)

warped = np.array(warped_debayered_calibrated_frames) / 255

# Display large grid of warped images so small offsets are visible
n = len(warped)
cols = min(4, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)
for i, ax in enumerate(axes):
    if i >= n:
        ax.axis('off')
        continue
    wimg = warped[i]
    # if multichannel, show as RGB, otherwise grayscale
    if wimg is None:
        ax.set_title(f'img{i}: None')
        ax.axis('off')
        continue
    if wimg.ndim == 3 and wimg.shape[2] == 3:
        disp = percentile_normalize(wimg, 1, 99.9)
        ax.imshow(disp[..., ::])
    else:
        arr = np.asarray(wimg).astype(float)
        pmin, pmax = np.percentile(arr, (1, 99))
        ax.imshow(np.clip((arr - pmin) / (pmax - pmin + 1e-12), 0, 1), cmap='gray')
    ax.set_title(f'warped img {i}')
    ax.axis('off')
plt.suptitle(f'Warped images into reference {REF_IDX}')
plt.tight_layout()
plt.show()

## 8. Résultat Final

Après toutes ces étapes, nous obtenons enfin notre image finale !

Comparez cette image avec les images brutes du début. Vous remarquerez :
- Une **nette amélioration du rapport signal/bruit**.
- La **disparition des défauts** (pixels chauds, vignettage).
- Des **détails beaucoup plus fins** dans les bras spiraux et le bulbe de la galaxie.

Ce pipeline traditionnel est la base de l'astrophotographie du ciel profond et permet de transformer des données bruitées en une image de qualité scientifique et esthétique.

In [ ]:
master_image = make_master_frame(warped_debayered_calibrated_frames)
master_image /= master_image.mean(axis=(0, 1))

In [ ]:
stretch = ImageNormalize(stretch=AsinhStretch(a=0.001))
plt.figure(figsize=(6, 6), dpi=300)
image = stretch(percentile_normalize(master_image, 60, 100))
plt.imshow(image - image.min())
plt.axis('off')